# Interactive Loss Function Plots

This notebook renders interactive Plotly figures for the memetic loss functions.

Use the controls below to tune temperatures and plotting ranges for both losses.

Available plots:
1. Raw dBm vs normalized score
2. Raw dBm vs normalized penalty
3. Penalty vs demand-weighted normalized softmin loss
4. Normalized score vs demand-weighted normalized softmin loss
5. Raw dBm vs demand-weighted normalized softmin loss
6. Raw dBm vs soft coverage loss

In [2]:
from pathlib import Path
import sys

import ipywidgets as widgets
from IPython.display import display

# Ensure the project root is importable when running from notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.plot_loss import (
    plot_normalization_curve,
    plot_penalty_curve,
    plot_softmin_loss_vs_penalty,
    plot_softmin_loss_vs_normalized_score,
    plot_softmin_loss_vs_raw_dbm,
    plot_soft_coverage_loss_curve,
)

In [ ]:
def _parse_float_tuple(text: str, label: str) -> tuple[float, ...]:
    values = []
    for part in text.split(","):
        part = part.strip()
        if not part:
            continue
        values.append(float(part))
    if not values:
        raise ValueError(f"{label} cannot be empty")
    return tuple(values)


current_figures = {}

plot_options = [
    "1) Raw dBm vs normalized score",
    "2) Raw dBm vs normalized penalty",
    "3) Penalty vs softmin loss",
    "4) Normalized score vs softmin loss",
    "5) Raw dBm vs softmin loss",
    "6) Raw dBm vs soft coverage loss",
]

softmin_temps = widgets.Text(
    value="0.5, 0.7, 1.0, 2.0, 5.0",
    description="SoftMin T",
    layout=widgets.Layout(width="360px"),
)
coverage_temps = widgets.Text(
    value="0.1, 0.3, 0.7, 1.0, 2.0",
    description="Coverage T",
    layout=widgets.Layout(width="360px"),
)

floor_dbm = widgets.FloatSlider(
    value=-120.0, min=-150.0, max=-40.0, step=1.0, description="floor dBm"
 )
ceil_dbm = widgets.FloatSlider(
    value=-70.0, min=-140.0, max=-20.0, step=1.0, description="ceil dBm"
 )
threshold_dbm = widgets.FloatSlider(
    value=-75.0, min=-140.0, max=-20.0, step=1.0, description="threshold"
 )

x_range = widgets.FloatRangeSlider(
    value=[-140.0, -40.0],
    min=-160.0, max=0.0, step=1.0,
    description="x dBm range",
    continuous_update=False,
    layout=widgets.Layout(width="550px"),
)
p_range = widgets.FloatRangeSlider(
    value=[-0.5, 3.0],
    min=-2.0, max=5.0, step=0.1,
    description="penalty range",
    continuous_update=False,
    layout=widgets.Layout(width="550px"),
)
s_range = widgets.FloatRangeSlider(
    value=[-2.0, 2.0],
    min=-4.0, max=4.0, step=0.1,
    description="score range",
    continuous_update=False,
    layout=widgets.Layout(width="550px"),
)

fixed_penalties = widgets.Text(
    value="0.2, 0.5, 1.0",
    description="fixed p",
    layout=widgets.Layout(width="320px"),
)
fixed_weights = widgets.Text(
    value="1.0, 3.0, 0.8",
    description="fixed w",
    layout=widgets.Layout(width="320px"),
)
varying_weight = widgets.FloatSlider(
    value=2.0, min=0.1, max=10.0, step=0.1, description="var w"
 )

num_points = widgets.IntSlider(
    value=400, min=100, max=1200, step=50, description="num points"
 )
selected_plots = widgets.SelectMultiple(
    options=plot_options,
    value=tuple(plot_options),
    description="Show",
    layout=widgets.Layout(width="380px", height="160px"),
)

status = widgets.HTML(value="")
plot_output = widgets.Output()
update_button = widgets.Button(description="Update plots", button_style="success")


def render_plots(_=None):
    global current_figures
    with plot_output:
        plot_output.clear_output(wait=True)
        status.value = ""
        try:
            st = _parse_float_tuple(softmin_temps.value, "SoftMin T")
            ct = _parse_float_tuple(coverage_temps.value, "Coverage T")
            fp = _parse_float_tuple(fixed_penalties.value, "fixed penalties")
            fw = _parse_float_tuple(fixed_weights.value, "fixed weights")
            if len(fp) != len(fw):
                raise ValueError("fixed penalties and fixed weights must have same length")
            if ceil_dbm.value <= floor_dbm.value:
                raise ValueError("ceil dBm must be greater than floor dBm")

            x_min, x_max = x_range.value
            p_min, p_max = p_range.value
            s_min, s_max = s_range.value

            figs = {
                plot_options[0]: plot_normalization_curve(
                    floor_dbm=floor_dbm.value,
                    ceil_dbm=ceil_dbm.value,
                    x_min_dbm=x_min,
                    x_max_dbm=x_max,
                    num_points=num_points.value,
                ),
                plot_options[1]: plot_penalty_curve(
                    floor_dbm=floor_dbm.value,
                    ceil_dbm=ceil_dbm.value,
                    x_min_dbm=x_min,
                    x_max_dbm=x_max,
                    num_points=num_points.value,
                ),
                plot_options[2]: plot_softmin_loss_vs_penalty(
                    temperatures=st,
                    fixed_penalties=fp,
                    fixed_weights=fw,
                    varying_weight=varying_weight.value,
                    p_min=p_min,
                    p_max=p_max,
                    num_points=num_points.value,
                ),
                plot_options[3]: plot_softmin_loss_vs_normalized_score(
                    temperatures=st,
                    fixed_penalties=fp,
                    fixed_weights=fw,
                    varying_weight=varying_weight.value,
                    s_min=s_min,
                    s_max=s_max,
                    num_points=num_points.value,
                ),
                plot_options[4]: plot_softmin_loss_vs_raw_dbm(
                    temperatures=st,
                    floor_dbm=floor_dbm.value,
                    ceil_dbm=ceil_dbm.value,
                    fixed_penalties=fp,
                    fixed_weights=fw,
                    varying_weight=varying_weight.value,
                    x_min_dbm=x_min,
                    x_max_dbm=x_max,
                    num_points=num_points.value,
                ),
                plot_options[5]: plot_soft_coverage_loss_curve(
                    threshold_dbm=threshold_dbm.value,
                    temperatures=ct,
                    x_min_dbm=x_min,
                    x_max_dbm=x_max,
                    num_points=num_points.value,
                ),
            }

            current_figures = {k: v for k, v in figs.items() if k in selected_plots.value}
            for name, fig in current_figures.items():
                print(name)
                fig.show()
        except Exception as exc:
            status.value = f"<b style='color:#b00'>Error:</b> {exc}"


update_button.on_click(render_plots)

controls_left = widgets.VBox([
    softmin_temps,
    coverage_temps,
    floor_dbm,
    ceil_dbm,
    threshold_dbm,
    num_points,
    varying_weight,
])
controls_middle = widgets.VBox([x_range, p_range, s_range])
controls_right = widgets.VBox([fixed_penalties, fixed_weights, selected_plots, update_button, status])

display(widgets.HBox([controls_left, controls_middle, controls_right]))
display(plot_output)

# Initial draw
render_plots()

Output()

In [4]:
# Optional: export currently displayed interactive HTML files
save_dir = PROJECT_ROOT / "tmp_results" / "loss_plots"
save_dir.mkdir(parents=True, exist_ok=True)

if not current_figures:
    raise RuntimeError("No figures available. Run Cell 3 first to generate plots.")

slug_map = {
    "1) Raw dBm vs normalized score": "01_normalization",
    "2) Raw dBm vs normalized penalty": "02_penalty",
    "3) Penalty vs softmin loss": "03_softmin_vs_penalty",
    "4) Normalized score vs softmin loss": "04_softmin_vs_normalized",
    "5) Raw dBm vs softmin loss": "05_softmin_vs_raw_dbm",
    "6) Raw dBm vs soft coverage loss": "06_soft_coverage",
}

for display_name, fig in current_figures.items():
    file_name = f"{slug_map[display_name]}.html"
    fig.write_html(save_dir / file_name, include_plotlyjs="cdn")

print(f"Saved {len(current_figures)} interactive plot(s) to: {save_dir}")

Saved 6 interactive plot(s) to: /home/hieule/research/reflector-position/tmp_results/loss_plots
